<a href="https://colab.research.google.com/github/sflores14/inspirastem2026-bioquimica-computacional/blob/main/day2/02_prediccion_reconocimiento_molecular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 2 | Predicción de reconocimiento molecular

**InspiraSTEM 2026 | Bioquímica Computacional Aplicada**

### Pregunta del día

> **Cómo podemos predecir cómo se une una molécula a una proteína, y qué hacemos cuando diferentes métodos no están de acuerdo?**

Ayer llegamos hasta:

**estructura -> pocket -> química -> hipótesis**

Hoy vamos a poner esa hipótesis a prueba con dos estrategias distintas:

**Boltz-2 -> GNINA -> interacciones -> comparación de evidencia**

**Boltz-2** es un modelo generativo biomolecular que recibe la secuencia de una proteína y la estructura química de un ligando para proponer un complejo 3D, estimar la confianza estructural y predecir afinidad.

**GNINA** es un programa de *molecular docking* que recibe una estructura 3D de proteína, un ligando y una región de búsqueda para generar poses y evaluarlas con una función de puntuación.

La meta no es que los dos métodos den exactamente el mismo número. La meta es aprender a comparar evidencia y decidir cuánto cambia nuestra hipótesis.


## 1. Dos métodos, dos formas de pensar

| | Boltz-2 | GNINA |
|---|---|---|
| Entrada principal | secuencia + ligando | estructura 3D + ligando |
| Región de búsqueda | no definimos un docking box | definimos un docking box |
| Pregunta | cómo podría verse el complejo? | qué poses son favorables dentro de esta región? |
| Resultado principal | complejo + confianza + afinidad predicha | poses + `minimizedAffinity` |

> ### Pregunta para tu equipo
> **Qué diferencia fundamental hay entre predecir un complejo desde secuencia + ligando y buscar una pose dentro de una estructura y región ya definidas?**


## 2. Instalar Boltz-2

Este notebook está preparado para Google Colab.

Para mantener compatibilidad con versiones recientes de Python usamos `boltz-community`, que conserva el comando `boltz` y permite ejecutar los modelos de Boltz-2 en Colab.

La instalación puede tardar unos minutos. Cuando termine, la celda confirmará que el comando de Boltz-2 está disponible.


In [ ]:
#@title Instalar Boltz-2

import sys
import subprocess
import importlib.metadata as metadata
import platform

BOLTZ_COMMUNITY_VERSION = "2.10.12"

print("Versión de Python:", platform.python_version())
print("Instalando Boltz-2 para este entorno de Colab...")

try:
    current = metadata.version("boltz-community")
except metadata.PackageNotFoundError:
    current = None

if current != BOLTZ_COMMUNITY_VERSION:
    result = subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            f"boltz-community=={BOLTZ_COMMUNITY_VERSION}"
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    if result.returncode != 0:
        print(result.stdout[-5000:])
        raise RuntimeError("La instalación de Boltz-2 no terminó correctamente.")

print("Versión instalada:", metadata.version("boltz-community"))

cli = subprocess.run(
    ["boltz", "--help"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

if cli.returncode != 0:
    print(cli.stdout[-2000:])
    raise RuntimeError("La instalación terminó, pero el comando boltz no está disponible.")

print("Boltz-2 está listo.")


## 3. Preparar el entorno base

Esta celda prepara los paquetes ligeros, descarga la estructura 4W51 y vuelve a calcular los pockets con P2Rank.

También define los tres ligandos que compararemos hoy:

- Benzene
- Toluene
- n-Propylbenzene


In [ ]:
#@title Preparar entorno base

import os, sys, json, time, tarfile, shutil, subprocess, importlib.util, urllib.request
from pathlib import Path

def ensure_package(package, import_name=None):
    import_name = import_name or package
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

for pkg, imp in [
    ("py3Dmol", "py3Dmol"),
    ("requests", "requests"),
    ("pyyaml", "yaml"),
]:
    ensure_package(pkg, imp)

import numpy as np
import pandas as pd
import requests
import yaml
import py3Dmol

from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from IPython.display import display, HTML

WORK = Path("/content/inspirastem_day2")
WORK.mkdir(exist_ok=True)

# T4 lysozyme L99A
PDB_ID = "4W51"
r = requests.get(f"https://files.rcsb.org/download/{PDB_ID}.pdb", timeout=60)
r.raise_for_status()
raw_pdb = r.text

protein_lines = [
    line for line in raw_pdb.splitlines()
    if line.startswith("ATOM") and len(line) > 21 and line[21] == "A"
]
protein_pdb = "\n".join(protein_lines + ["END"]) + "\n"
PROTEIN_FILE = WORK / "4W51_protein.pdb"
PROTEIN_FILE.write_text(protein_pdb)

# P2Rank 2.5.1
if shutil.which("java") is None:
    subprocess.check_call(["apt-get", "update", "-qq"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(
        ["apt-get", "install", "-y", "-qq", "openjdk-17-jre-headless"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

P2_DIR = WORK / "p2rank_2.5.1"
if not P2_DIR.exists():
    archive = WORK / "p2rank_2.5.1.tar.gz"
    urllib.request.urlretrieve(
        "https://github.com/rdk/p2rank/releases/download/2.5.1/p2rank_2.5.1.tar.gz",
        archive
    )
    P2_DIR.mkdir(exist_ok=True)
    with tarfile.open(archive, "r:gz") as tf:
        tf.extractall(P2_DIR)

prank_candidates = list(P2_DIR.rglob("prank"))
if not prank_candidates:
    raise RuntimeError("No se encontró P2Rank.")
PRANK = prank_candidates[0]
os.chmod(PRANK, 0o755)

P2_OUT = WORK / "p2rank_output"
P2_OUT.mkdir(exist_ok=True)

cmd = [
    str(PRANK), "predict",
    "-f", str(PROTEIN_FILE.resolve()),
    "-o", str(P2_OUT.resolve()),
    "-visualizations", "0"
]
result = subprocess.run(
    cmd, cwd=str(PRANK.parent),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True
)
if result.returncode != 0:
    print(result.stdout[-3000:])
    raise RuntimeError("P2Rank no terminó correctamente.")

prediction_files = list(P2_OUT.rglob("*_predictions.csv"))
if not prediction_files:
    raise FileNotFoundError("No se encontró la tabla de P2Rank.")

pockets = pd.read_csv(prediction_files[0], skipinitialspace=True)
pockets.columns = [c.strip() for c in pockets.columns]

ligands = {
    "Benzene": "c1ccccc1",
    "Toluene": "Cc1ccccc1",
    "n-Propylbenzene": "CCCc1ccccc1"
}
slug = {
    "Benzene": "benzene",
    "Toluene": "toluene",
    "n-Propylbenzene": "n_propylbenzene"
}

print("Entorno base listo.")
print(f"P2Rank encontró {len(pockets)} pockets.")


## 4. Recuperar la decisión del Día 1 y visualizar el pocket

Selecciona el mismo pocket que tu grupo eligió ayer.

### Parámetros importantes

**`pocket_rank`**
Indica cuál pocket de P2Rank usaremos. Usa el pocket que tu grupo seleccionó ayer. Para probar el notebook, el valor inicial recomendado es **2**.

**`box_size`**
Es la longitud de cada lado del cubo de búsqueda en Angstroms. El box debe cubrir la cavidad y dejar espacio para que el ligando explore distintas orientaciones.

- Valor inicial recomendado: **14 Å**
- Muy pequeño: puede excluir poses posibles.
- Muy grande: aumenta innecesariamente el espacio de búsqueda.
- Si duplicamos la longitud de cada lado, el volumen del box aumenta aproximadamente 8 veces.

En la visualización:
- proteína: gris
- residuos dentro del box: naranja
- centro de P2Rank: esfera naranja
- docking box: amarillo

> ### Pregunta para tu equipo
> **El box cubre la cavidad que eligieron y deja suficiente espacio para los ligandos? Qué cambiaría si el box fuera mucho más grande?**


In [ ]:
#@title Seleccionar pocket y visualizar el docking box

pocket_rank = 2 #@param {type:"slider", min:1, max:5, step:1}
box_size = 14.0 #@param {type:"slider", min:10.0, max:22.0, step:1.0}

if pocket_rank > len(pockets):
    raise ValueError("Ese pocket no existe en la salida de P2Rank.")

pocket = pockets.iloc[pocket_rank - 1]
center = np.array([
    float(pocket["center_x"]),
    float(pocket["center_y"]),
    float(pocket["center_z"])
])

# Identificar residuos con al menos un átomo dentro del box.
half = box_size / 2.0
pocket_residues = set()

for line in protein_pdb.splitlines():
    if not line.startswith("ATOM") or len(line) < 54:
        continue
    try:
        x = float(line[30:38])
        y = float(line[38:46])
        z = float(line[46:54])
        resi = int(line[22:26])
    except ValueError:
        continue

    if (
        abs(x - center[0]) <= half
        and abs(y - center[1]) <= half
        and abs(z - center[2]) <= half
    ):
        pocket_residues.add(resi)

pocket_residues = sorted(pocket_residues)

view = py3Dmol.view(width=850, height=560)
view.setBackgroundColor("white")
view.addModel(protein_pdb, "pdb")
view.setStyle({"chain":"A"}, {"cartoon":{"color":"lightgray"}})

if pocket_residues:
    view.setStyle(
        {"chain":"A", "resi":pocket_residues},
        {
            "stick":{"colorscheme":"orangeCarbon", "radius":0.22},
            "sphere":{"colorscheme":"orangeCarbon", "scale":0.14}
        }
    )
    view.addSurface(
        py3Dmol.VDW,
        {"opacity":0.22, "color":"#D9D9D9"},
        {"chain":"A", "resi":pocket_residues}
    )

view.addSphere({
    "center":{"x":float(center[0]), "y":float(center[1]), "z":float(center[2])},
    "radius":0.8,
    "color":"orange",
    "opacity":0.95
})

view.addBox({
    "center":{"x":float(center[0]), "y":float(center[1]), "z":float(center[2])},
    "dimensions":{"w":box_size, "h":box_size, "d":box_size},
    "color":"yellow",
    "opacity":0.18
})

if pocket_residues:
    view.zoomTo({"chain":"A", "resi":pocket_residues})
else:
    view.zoomTo()

view.show()

print(f"Pocket seleccionado: {pocket_rank}")
print(f"Centro del box: ({center[0]:.2f}, {center[1]:.2f}, {center[2]:.2f})")
print(f"Tamaño del box: {box_size:.1f} Å por lado")
print("Residuos dentro del box:", pocket_residues if pocket_residues else "ninguno")


### Antes de continuar

Mira el pocket desde varios ángulos.

> **Qué residuos forman la cavidad? El box incluye la región que el grupo quería estudiar desde el Día 1?**

Si la respuesta es no, ajusta primero el pocket o el tamaño del box. El docking dependerá de esta decisión.


# Parte I | Predicción estructural con Boltz-2

**Boltz-2** recibe la secuencia de la proteína y la estructura química del ligando y genera una hipótesis tridimensional del complejo, además de métricas de confianza y una predicción de afinidad.

Esto es diferente a docking: no estamos insertando el ligando dentro de la estructura experimental 4W51 y no le estamos indicando el pocket que escogimos ayer.

> ### Pregunta para tu equipo
> **Boltz-2 sabe cuál pocket escogimos ayer? Qué información sí le estamos dando?**


## 5. Elegir una predicción para la demostración en vivo

En CPU, una predicción puede tardar alrededor de 15 minutos en este ejercicio. Cada equipo puede escoger **un solo ligando** para la demostración.

Los resultados completos de los tres ligandos ya fueron calculados y se cargarán en la sección 7.

> ### Pregunta para tu equipo
> **Cuál de los tres ligandos quieren predecir en vivo y qué esperan observar antes de correr el modelo?**


In [ ]:
#@title Elegir ligando y crear UN input de Boltz-2

# Secuencia de T4 lysozyme L99A, 164 aa
protein_sequence = (
    "MNIFEMLRIDERLRLKIYKDTEGYYTIGIGHLLTKSPSLNAAKSELDKAIGRNCNGVITKDEAE"
    "KLFNQDVDAAVRGILRNAKLKPVYDSLDAVRRCAAINMVFQMGETGVAGFTNSLRMLQQKRWDEA"
    "AVNLAKSIWYNQTPNRAKRVITTFRTGTWDAYKNL"
)

live_ligand = "Benzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]

print("Residuo 99 en la secuencia usada:", protein_sequence[98])
print("Longitud de la proteína:", len(protein_sequence), "aa")
print("Ligando elegido para la demostración:", live_ligand)

LIVE_BOLTZ_INPUTS = WORK / "boltz_live_input"
if LIVE_BOLTZ_INPUTS.exists():
    shutil.rmtree(LIVE_BOLTZ_INPUTS)
LIVE_BOLTZ_INPUTS.mkdir(exist_ok=True)

name = live_ligand
smi = ligands[name]
data = {
    "version": 1,
    "sequences": [
        {"protein": {"id": "A", "sequence": protein_sequence}},
        {"ligand": {"id": "B", "smiles": smi}}
    ],
    "properties": [{"affinity": {"binder": "B"}}]
}

live_yaml = LIVE_BOLTZ_INPUTS / f"{slug[name]}.yaml"
live_yaml.write_text(yaml.safe_dump(data, sort_keys=False))
print("Archivo de entrada creado:", live_yaml.name)


## 6. Ejecutar o saltar la demostración de Boltz-2

La demostración es opcional.

- Si hay tiempo, cambia `ejecutar_demo_boltz` a `True`.
- Si quieren avanzar directamente al análisis, déjalo en `False` y continúen con la sección 7.
- En CPU, una predicción puede tardar cerca de **15 minutos**.
- No es necesario esperar una predicción en vivo para completar el resto del notebook.

Mientras corre, si deciden hacerla:

> ### Pregunta para tu equipo
> **Qué información está usando Boltz-2 para construir esta predicción y qué información del Día 1 no está usando?**


In [ ]:
#@title Ejecutar o saltar UNA predicción de Boltz-2

import time
import torch
import subprocess

ejecutar_demo_boltz = False #@param {type:"boolean"}
acelerador = "Auto" #@param ["Auto", "gpu", "cpu"]

if acelerador == "Auto":
    accelerator = "gpu" if torch.cuda.is_available() else "cpu"
else:
    accelerator = acelerador

print("Acelerador seleccionado:", accelerator)

if not ejecutar_demo_boltz:
    print("Demostración omitida.")
    print("Continúa directamente con la sección 7 y usa los resultados precomputados.")
else:
    if accelerator == "cpu":
        print("La predicción en CPU puede tardar cerca de 15 minutos.")

    # Ajustes rápidos y fijos para la demostración del workshop.
    recycling_steps = 1
    sampling_steps = 50
    diffusion_samples = 1
    sampling_steps_affinity = 50
    diffusion_samples_affinity = 1
    max_msa_seqs = 1024

    LIVE_BOLTZ_OUT = WORK / "boltz_live_results"

    cmd = [
        "boltz", "predict", str(LIVE_BOLTZ_INPUTS),
        "--out_dir", str(LIVE_BOLTZ_OUT),
        "--use_msa_server",
        "--accelerator", accelerator,
        "--output_format", "pdb",
        "--num_workers", "0",
        "--recycling_steps", str(recycling_steps),
        "--sampling_steps", str(sampling_steps),
        "--diffusion_samples", str(diffusion_samples),
        "--sampling_steps_affinity", str(sampling_steps_affinity),
        "--diffusion_samples_affinity", str(diffusion_samples_affinity),
        "--max_msa_seqs", str(max_msa_seqs),
        "--no_kernels"
    ]

    print(f"Ejecutando la predicción en vivo para {live_ligand}...")
    t0 = time.time()
    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    elapsed = time.time() - t0

    if result.returncode != 0:
        print(result.stdout[-5000:])
        raise RuntimeError("Boltz-2 no terminó correctamente.")

    print(f"Boltz-2 terminó en {elapsed/60:.1f} minutos.")


## 7. Cargar los resultados precomputados de los tres ligandos

Para comparar los tres ligandos sin esperar múltiples predicciones, descargaremos resultados calculados previamente para:

- Benzene
- Toluene
- n-Propylbenzene

Esto separa dos tareas:

1. **Demostración:** ver cómo se genera una predicción.
2. **Análisis:** interpretar y comparar los resultados.

> ### Pregunta para tu equipo
> **Por qué guardar y reutilizar resultados computacionales puede ser útil para reproducibilidad y eficiencia?**


In [ ]:
#@title Descargar resultados precomputados de Boltz-2

from pathlib import Path
import urllib.request
import zipfile
import shutil

ZIP_URL = "https://raw.githubusercontent.com/sflores14/inspirastem2026-bioquimica-computacional/main/day2/boltz_results.zip"
ZIP_PATH = Path("/content/boltz_results.zip")
EXTRACT_DIR = Path("/content/boltz_results")

if not ZIP_PATH.exists():
    print("Descargando resultados precomputados de Boltz-2...")
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
else:
    print("El archivo de resultados ya está disponible.")

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

print("Resultados precomputados listos.")
print(f"Carpeta de trabajo: {EXTRACT_DIR}")


## 8. Interpretar confianza y afinidad de Boltz-2

Vamos a mirar tres ideas diferentes.

### Confianza estructural del complejo

`complex_plddt` se reporta originalmente entre 0 y 1. En este notebook lo multiplicamos por 100 para mostrarlo en una escala **0 a 100 tipo porcentaje**.

Un valor alto significa que el modelo tiene mayor confianza en la estructura que propone. No es una probabilidad experimental de que el complejo sea correcto.

### Probabilidad de unión

`affinity_probability_binary` también se reporta entre 0 y 1. Aquí la mostramos como **porcentaje**.

Esta métrica estima la probabilidad de que el ligando se comporte como binder dentro del modelo.

### Afinidad y energía libre estimada

Boltz-2 reporta `affinity_pred_value` como:

`y = log10(IC50 en µM)`

Para tener una cantidad en kcal/mol que podamos comparar de forma aproximada con el score de docking, haremos una conversión termodinámica bajo una suposición fuerte:

**Kd aproximadamente igual a IC50**

Primero:

`IC50(M) = 10^y x 10^-6`

Luego:

`Delta G° estimada = R T ln(Kd / 1 M)`

Usaremos una temperatura inicial recomendada de **298.15 K**, aproximadamente 25 °C.

A 298.15 K:

`Delta G° estimada ≈ 1.364 x (y - 6) kcal/mol`

Valores más negativos representan una unión estimada más favorable bajo esta aproximación.

**Importante:** IC50 no es exactamente Kd, y esta conversión no convierte la predicción de Boltz-2 en una medición experimental de energía libre.

> ### Pregunta para tu equipo
> **Confianza estructural, probabilidad de unión y afinidad predicha significan lo mismo?**


In [ ]:
#@title Resumir Boltz-2 y convertir afinidad a energía libre estimada

import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

temperatura_K = 298.15 #@param {type:"number"}
R_kcal = 0.00198720425864083  # kcal mol^-1 K^-1

# Encontrar automáticamente la carpeta "predictions" dentro del ZIP.
prediction_roots = [
    p for p in EXTRACT_DIR.rglob("predictions")
    if p.is_dir() and all((p / slug[name]).exists() for name in ligands)
]

if not prediction_roots:
    raise FileNotFoundError("No se encontró la carpeta de predicciones dentro del ZIP.")

PREDICTIONS_DIR = prediction_roots[0]

boltz_structures = {}
boltz_rows = []

for name in ligands:
    s = slug[name]
    pred_dir = PREDICTIONS_DIR / s

    structures = list(pred_dir.glob("*_model_0.pdb"))
    confidences = list(pred_dir.glob("confidence_*_model_0.json"))
    affinities = list(pred_dir.glob("affinity_*.json"))

    if not structures or not confidences or not affinities:
        print(f"Faltan archivos para {name} en {pred_dir}.")
        continue

    structure_path = structures[0]
    boltz_structures[name] = structure_path

    conf = json.loads(confidences[0].read_text())
    aff = json.loads(affinities[0].read_text())

    y = float(aff.get("affinity_pred_value", np.nan))
    ic50_um = 10 ** y if np.isfinite(y) else np.nan
    ic50_M = ic50_um * 1e-6 if np.isfinite(ic50_um) else np.nan

    # Aproximación para fines comparativos: Kd aproximadamente igual a IC50.
    delta_g = (
        R_kcal * temperatura_K * np.log(ic50_M)
        if np.isfinite(ic50_M) and ic50_M > 0
        else np.nan
    )

    complex_plddt_pct = 100 * float(conf.get("complex_plddt", np.nan))
    binder_pct = 100 * float(aff.get("affinity_probability_binary", np.nan))

    boltz_rows.append({
        "Ligando": name,
        "Confianza complejo (%)": round(complex_plddt_pct, 1),
        "Probabilidad binder (%)": round(binder_pct, 1),
        "log10(IC50 µM)": round(y, 3),
        "IC50 pred. (µM)": round(ic50_um, 3),
        "Delta G estimada (kcal/mol)": round(delta_g, 3)
    })

boltz_df = pd.DataFrame(boltz_rows)
display(boltz_df)

plt.figure(figsize=(7, 4))
plt.bar(
    boltz_df["Ligando"],
    boltz_df["Delta G estimada (kcal/mol)"]
)
plt.ylabel("Delta G estimada (kcal/mol)")
plt.xlabel("Ligando")
plt.title(f"Boltz-2: energía libre estimada a {temperatura_K:.2f} K")
plt.xticks(rotation=15)
plt.axhline(0, linewidth=0.8)
plt.show()

print("Recuerda: esta Delta G usa la aproximación Kd aproximadamente igual a IC50.")


## 9. Explorar una predicción de Boltz-2 en 3D

Antes de mirar GNINA, observa una de las estructuras predichas por Boltz-2.

En la visualización:
- proteína: gris
- ligando de Boltz-2: verde

> ### Pregunta para tu equipo
> **Dónde colocó Boltz-2 el ligando? Está enterrado o expuesto? Se parece a la cavidad que eligieron en el Día 1?**


In [ ]:
#@title Explorar una predicción de Boltz-2

boltz_ligand = "Benzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]

boltz_pdb = boltz_structures[boltz_ligand].read_text()

view = py3Dmol.view(width=850, height=560)
view.setBackgroundColor("white")
view.addModel(boltz_pdb, "pdb")

view.setStyle(
    {"chain":"A"},
    {"cartoon":{"color":"lightgray"}}
)

view.setStyle(
    {"chain":"B"},
    {
        "stick":{"colorscheme":"greenCarbon", "radius":0.28},
        "sphere":{"colorscheme":"greenCarbon", "scale":0.22}
    }
)

view.addSurface(
    py3Dmol.VDW,
    {"opacity":0.12, "color":"#D9D9D9"},
    {"chain":"A"}
)

view.zoomTo({"chain":"B"})
view.show()

print("Ligando de Boltz-2: verde.")


### Antes de pasar a docking

Mueve el complejo y discute:

- El ligando aparece enterrado o expuesto?
- El modelo colocó el ligando cerca del pocket que escogieron ayer?
- Qué residuos parecen estar cerca?
- Qué observación les haría desconfiar de esta pose?

No compares coordenadas directamente con GNINA todavía. Primero tendremos que alinear las proteínas.


# Parte II | Molecular docking con GNINA

**GNINA** recibe una estructura 3D de proteína, un ligando y una región de búsqueda. Después explora orientaciones posibles del ligando dentro de esa región y asigna un score a cada pose.

En nuestro caso:
- proteína experimental: **4W51**
- región de búsqueda: el docking box definido a partir del pocket de P2Rank
- score que analizaremos: **`minimizedAffinity`**

> ### Pregunta para tu equipo
> **Qué información le estamos dando a GNINA que no le dimos a Boltz-2?**


## 10. Instalar GNINA y preparar los ligandos

Para evitar conflictos de dependencias en Google Colab, este notebook usa el binario genérico de **GNINA v1.0.3** y ejecuta el docking por **CPU**.

Esto funciona tanto si el runtime de Colab es CPU como si es GPU, porque esta parte del notebook usa CPU de forma intencional.

También generaremos una conformación 3D inicial para cada ligando antes del docking.


In [ ]:
#@title Instalar GNINA y preparar ligandos

import os
import sys
import subprocess
import urllib.request
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem

GNINA = WORK / "gnina"
GNINA_URL = "https://github.com/gnina/gnina/releases/download/v1.0.3/gnina"

if not GNINA.exists():
    print("Descargando GNINA v1.0.3...")
    urllib.request.urlretrieve(GNINA_URL, GNINA)
    os.chmod(GNINA, 0o755)
else:
    print("GNINA ya está descargado.")

version = subprocess.run(
    [str(GNINA), "--version"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

if version.returncode != 0:
    print(version.stdout[-2000:])
    raise RuntimeError("GNINA no pudo iniciarse correctamente.")

print("GNINA está listo.")
if version.stdout:
    print("Versión reportada:", version.stdout.strip().splitlines()[0])

GNINA_LIGANDS = WORK / "gnina_ligands"
GNINA_LIGANDS.mkdir(exist_ok=True)

gnina_ligand_files = {}

for name, smi in ligands.items():
    mol = Chem.AddHs(Chem.MolFromSmiles(smi))
    params = AllChem.ETKDGv3()
    params.randomSeed = 2026

    if AllChem.EmbedMolecule(mol, params) != 0:
        raise RuntimeError(f"No se pudo generar una conformación 3D para {name}.")

    AllChem.MMFFOptimizeMolecule(mol)
    mol.SetProp("_Name", name)

    path = GNINA_LIGANDS / f"{slug[name]}.sdf"
    writer = Chem.SDWriter(str(path))
    writer.write(mol)
    writer.close()

    gnina_ligand_files[name] = path

print("Los tres ligandos están preparados para docking.")


## 11. Configurar y ejecutar docking

Ahora controlaremos explícitamente la búsqueda.

### `box_size`

Es la longitud de cada lado del docking box en Angstroms. Ya la elegimos en la sección 4.

- Valor inicial recomendado: **14 Å**
- Debe cubrir la cavidad y dejar margen para distintas orientaciones.
- Un box innecesariamente grande aumenta el espacio que GNINA debe explorar.

### `exhaustiveness`

Controla cuánto esfuerzo dedica GNINA a explorar el espacio de búsqueda.

- Valor inicial recomendado: **8**
- Un valor mayor explora más y normalmente tarda más.
- Para este workshop, 8 es un buen punto de partida.

### `num_modes`

Es el número máximo de poses que queremos guardar y revisar para cada ligando.

- Valor inicial recomendado: **5**
- Más modos significa más poses alternativas para inspeccionar.

### `cpu_threads`

Es el número de hilos de CPU usados durante el cálculo.

- Valor inicial recomendado: **2**
- El notebook nunca usará más hilos de los que tenga disponibles el runtime.

> ### Pregunta para tu equipo
> **Qué creen que pasaría con el tiempo de cálculo y la búsqueda si aumentamos mucho el box o el exhaustiveness?**


In [ ]:
#@title Docking de los tres ligandos

exhaustiveness = 8 #@param {type:"slider", min:2, max:32, step:2}
num_modes = 5 #@param {type:"slider", min:1, max:10, step:1}
cpu_threads = 2 #@param {type:"slider", min:1, max:8, step:1}

cpu_disponibles = os.cpu_count() or 1
cpu_usados = min(int(cpu_threads), int(cpu_disponibles))

print(f"Hilos de CPU disponibles: {cpu_disponibles}")
print(f"Hilos de CPU que usará GNINA: {cpu_usados}")
print(f"Exhaustiveness: {exhaustiveness}")
print(f"Número máximo de poses por ligando: {num_modes}")
print(f"Tamaño del box: {box_size:.1f} Å por lado")

GNINA_OUT = WORK / "gnina_results"
GNINA_OUT.mkdir(exist_ok=True)

gnina_outputs = {}

for name in ligands:
    out_sdf = GNINA_OUT / f"{slug[name]}_docked.sdf"

    cmd = [
        str(GNINA),
        "-r", str(PROTEIN_FILE),
        "-l", str(gnina_ligand_files[name]),
        "--center_x", str(float(center[0])),
        "--center_y", str(float(center[1])),
        "--center_z", str(float(center[2])),
        "--size_x", str(float(box_size)),
        "--size_y", str(float(box_size)),
        "--size_z", str(float(box_size)),
        "--scoring", "vinardo",
        "--cnn_scoring", "none",
        "--exhaustiveness", str(int(exhaustiveness)),
        "--num_modes", str(int(num_modes)),
        "--cpu", str(cpu_usados),
        "--seed", "2026",
        "--no_gpu",
        "-q",
        "-o", str(out_sdf)
    ]

    print(f"Ejecutando docking para {name}...")
    run = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    if run.returncode != 0:
        print(run.stdout[-3000:])
        raise RuntimeError(f"GNINA no terminó correctamente para {name}.")

    gnina_outputs[name] = out_sdf
    print(f"Docking terminado para {name}.")

print("Docking terminado para los tres ligandos.")


## 12. Comparar `minimizedAffinity` con la energía libre estimada de Boltz-2

Para GNINA analizaremos **`minimizedAffinity`**, reportado en **kcal/mol**.

- Valores más negativos representan poses más favorables dentro de la función de puntuación.
- No es una medición experimental de energía libre.
- No debemos interpretar una diferencia pequeña como una verdad absoluta.

Para Boltz-2 ya convertimos la predicción de IC50 a una **Delta G estimada en kcal/mol** usando:

- temperatura = 298.15 K
- aproximación Kd aproximadamente igual a IC50

Ahora ambas cantidades están expresadas en kcal/mol, así que podemos compararlas visualmente de forma aproximada.

**Todavía no son la misma cantidad física:** una proviene de una función de docking y la otra de una conversión aproximada de una predicción de IC50.

> ### Pregunta para tu equipo
> **Boltz-2 y GNINA producen la misma tendencia para los tres ligandos? Qué conclusión parece más robusta y cuál depende del método?**


In [ ]:
#@title Resumir minimizedAffinity y comparar con Boltz-2

gnina_best_mols = {}
gnina_rows = []

def get_minimized_affinity(mol):
    props = mol.GetPropsAsDict()

    if "minimizedAffinity" in props:
        return float(props["minimizedAffinity"])

    for key, value in props.items():
        if key.lower() == "minimizedaffinity":
            return float(value)

    return np.nan

for name, path in gnina_outputs.items():
    supplier = Chem.SDMolSupplier(str(path), removeHs=False)
    poses = [m for m in supplier if m is not None]

    if not poses:
        raise RuntimeError(f"No se pudieron leer poses para {name}.")

    scored = [
        (get_minimized_affinity(m), i, m)
        for i, m in enumerate(poses, 1)
    ]
    finite = [x for x in scored if np.isfinite(x[0])]

    if not finite:
        raise RuntimeError(f"No se encontró minimizedAffinity para {name}.")

    score, pose_num, best_mol = min(finite, key=lambda x: x[0])

    gnina_best_mols[name] = best_mol
    gnina_rows.append({
        "Ligando": name,
        "Mejor pose": pose_num,
        "minimizedAffinity (kcal/mol)": round(float(score), 3)
    })

gnina_df = pd.DataFrame(gnina_rows)
display(gnina_df)

plt.figure(figsize=(7, 4))
plt.bar(
    gnina_df["Ligando"],
    gnina_df["minimizedAffinity (kcal/mol)"]
)
plt.ylabel("minimizedAffinity (kcal/mol)")
plt.xlabel("Ligando")
plt.title("GNINA: score de la mejor pose")
plt.xticks(rotation=15)
plt.axhline(0, linewidth=0.8)
plt.show()

comparison_energy = boltz_df[
    ["Ligando", "Delta G estimada (kcal/mol)"]
].merge(
    gnina_df[
        ["Ligando", "minimizedAffinity (kcal/mol)"]
    ],
    on="Ligando",
    how="inner"
)

display(comparison_energy)

x = np.arange(len(comparison_energy))
width = 0.36

plt.figure(figsize=(8, 4.5))
plt.bar(
    x - width/2,
    comparison_energy["Delta G estimada (kcal/mol)"],
    width,
    label="Boltz-2: Delta G estimada",
    color="#2E7D32"
)
plt.bar(
    x + width/2,
    comparison_energy["minimizedAffinity (kcal/mol)"],
    width,
    label="GNINA: minimizedAffinity",
    color="#EF6C00"
)
plt.xticks(x, comparison_energy["Ligando"], rotation=15)
plt.ylabel("kcal/mol")
plt.xlabel("Ligando")
plt.title("Comparación aproximada de las dos métricas")
plt.axhline(0, linewidth=0.8)
plt.legend()
plt.show()

boltz_rank = boltz_df.sort_values(
    "Delta G estimada (kcal/mol)"
)["Ligando"].tolist()

gnina_rank = gnina_df.sort_values(
    "minimizedAffinity (kcal/mol)"
)["Ligando"].tolist()

ranking_df = pd.DataFrame({
    "Posición": [1, 2, 3],
    "Boltz-2": boltz_rank,
    "GNINA": gnina_rank
})

display(ranking_df)

print("Valores más negativos se interpretan como más favorables dentro de cada método.")
print("La comparación numérica entre métodos es aproximada, no una equivalencia experimental.")


## 13. Explorar una pose de GNINA en 3D

Ahora observa la mejor pose de GNINA para uno de los ligandos.

En la visualización:
- proteína: gris
- ligando de GNINA: naranja
- docking box: amarillo

> ### Pregunta para tu equipo
> **La pose está dentro de la cavidad que eligieron? Qué residuos parecen rodear al ligando? La orientación les parece plausible?**


In [ ]:
#@title Explorar una pose de GNINA

gnina_ligand = "n-Propylbenzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]

mol = gnina_best_mols[gnina_ligand]
sdf_block = Chem.MolToMolBlock(mol)

view = py3Dmol.view(width=850, height=560)
view.setBackgroundColor("white")
view.addModel(protein_pdb, "pdb")
view.setStyle(
    {"model":0},
    {"cartoon":{"color":"lightgray"}}
)
view.addSurface(
    py3Dmol.VDW,
    {"opacity":0.12, "color":"#D9D9D9"},
    {"model":0}
)

view.addModel(sdf_block, "sdf")
view.setStyle(
    {"model":1},
    {
        "stick":{"colorscheme":"orangeCarbon", "radius":0.28},
        "sphere":{"colorscheme":"orangeCarbon", "scale":0.22}
    }
)

view.addBox({
    "center":{"x":float(center[0]), "y":float(center[1]), "z":float(center[2])},
    "dimensions":{"w":float(box_size), "h":float(box_size), "d":float(box_size)},
    "color":"yellow",
    "opacity":0.10
})

view.zoomTo({"model":1})
view.show()

print("Ligando de GNINA: naranja.")
print("Docking box: amarillo.")


# Parte III | Comparar las hipótesis estructurales

## 14. Alinear Boltz-2 con 4W51

Boltz-2 y GNINA no parten del mismo sistema de coordenadas. Antes de superponer los ligandos, alinearemos la proteína predicha por Boltz-2 con la estructura experimental 4W51 usando átomos Cα equivalentes.

### Qué es RMSD?

**RMSD** significa *root-mean-square deviation* o desviación cuadrática media.

Aquí mide, en Angstroms, cuánto se separan en promedio los Cα equivalentes de las dos proteínas después del mejor alineamiento.

- RMSD menor: las dos estructuras proteicas quedan más parecidas después de alinearlas.
- RMSD mayor: hay más diferencias estructurales.

En esta sección el RMSD es del **alineamiento de la proteína**. No nos dice cuál pose del ligando es correcta.

> ### Pregunta para tu equipo
> **Por qué necesitamos alinear las proteínas antes de comparar las posiciones de los ligandos?**


In [ ]:
#@title Alinear Boltz con 4W51

def ca_coordinates(pdb_text, chain="A"):
    coords = {}
    for line in pdb_text.splitlines():
        if not line.startswith(("ATOM", "HETATM")):
            continue
        if len(line) < 54:
            continue
        if line[21].strip() != chain:
            continue
        if line[12:16].strip() != "CA":
            continue
        try:
            resi = int(line[22:26])
            xyz = np.array([
                float(line[30:38]),
                float(line[38:46]),
                float(line[46:54])
            ])
            coords[resi] = xyz
        except:
            pass
    return coords

def align_pdb_text(mobile_text, reference_text, chain="A"):
    mob_ca = ca_coordinates(mobile_text, chain)
    ref_ca = ca_coordinates(reference_text, chain)
    common = sorted(set(mob_ca) & set(ref_ca))

    if len(common) < 10:
        raise RuntimeError("No hay suficientes Cα para alinear.")

    P = np.array([mob_ca[i] for i in common])
    Q = np.array([ref_ca[i] for i in common])

    Pc = P.mean(axis=0)
    Qc = Q.mean(axis=0)

    P0 = P - Pc
    Q0 = Q - Qc

    C = P0.T @ Q0
    U, S, Vt = np.linalg.svd(C)
    R = U @ Vt

    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = U @ Vt

    t = Qc - Pc @ R

    fitted = P @ R + t
    rmsd = np.sqrt(np.mean(np.sum((fitted - Q) ** 2, axis=1)))

    new_lines = []
    for line in mobile_text.splitlines():
        if line.startswith(("ATOM", "HETATM")) and len(line) >= 54:
            try:
                xyz = np.array([
                    float(line[30:38]),
                    float(line[38:46]),
                    float(line[46:54])
                ])
                xyz2 = xyz @ R + t
                line = (
                    line[:30]
                    + f"{xyz2[0]:8.3f}{xyz2[1]:8.3f}{xyz2[2]:8.3f}"
                    + line[54:]
                )
            except:
                pass
        new_lines.append(line)

    return "\n".join(new_lines) + "\n", rmsd

aligned_boltz = {}
alignment_rmsd = {}

for name, path in boltz_structures.items():
    aligned_text, rmsd = align_pdb_text(path.read_text(), protein_pdb)
    aligned_boltz[name] = aligned_text
    alignment_rmsd[name] = rmsd

pd.DataFrame({
    "Ligando": list(alignment_rmsd),
    "RMSD de alineamiento Cα (Å)": [
        round(alignment_rmsd[x], 3) for x in alignment_rmsd
    ]
})


## 15. Superponer Boltz-2 y GNINA

Después del alineamiento podemos ver las dos hipótesis en el mismo sistema de coordenadas.

Colores:
- Boltz-2: verde
- GNINA: naranja
- proteína experimental: gris

> ### Pregunta para tu equipo
> **Las dos poses ocupan la misma región? Tienen una orientación parecida? Alguna parece chocar con la proteína?**


In [ ]:
#@title Superponer Boltz-2 y GNINA

compare_ligand = "n-Propylbenzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]
show_surface = True #@param {type:"boolean"}

boltz_aligned_text = aligned_boltz[compare_ligand]
gnina_block = Chem.MolToMolBlock(gnina_best_mols[compare_ligand])

view = py3Dmol.view(width=850, height=580)
view.setBackgroundColor("white")

# Receptor experimental
view.addModel(protein_pdb, "pdb")
view.setStyle(
    {"model":0},
    {"cartoon":{"color":"lightgray", "opacity":0.35}}
)

if show_surface:
    view.addSurface(
        py3Dmol.VDW,
        {"opacity":0.12, "color":"#D9D9D9"},
        {"model":0}
    )

# Complejo alineado de Boltz-2
view.addModel(boltz_aligned_text, "pdb")
view.setStyle(
    {"model":1, "chain":"A"},
    {"cartoon":{"color":"lightgray", "opacity":0.08}}
)
view.setStyle(
    {"model":1, "chain":"B"},
    {
        "stick":{"colorscheme":"greenCarbon", "radius":0.28},
        "sphere":{"colorscheme":"greenCarbon", "scale":0.20}
    }
)

# Pose de GNINA
view.addModel(gnina_block, "sdf")
view.setStyle(
    {"model":2},
    {
        "stick":{"colorscheme":"orangeCarbon", "radius":0.28},
        "sphere":{"colorscheme":"orangeCarbon", "scale":0.20}
    }
)

view.zoomTo({"model":2})
view.show()

print("Boltz-2: verde.")
print("GNINA: naranja.")


### Mira antes de calcular contactos

Compara visualmente:

- Las dos poses ocupan la misma región?
- Tienen una orientación parecida?
- Alguna parece chocar con la proteína?
- Cambia la respuesta entre Benzene, Toluene y n-Propylbenzene?

**Que dos métodos estén de acuerdo no demuestra que la pose sea verdadera. Que estén en desacuerdo tampoco significa que uno de los métodos sea inútil.**


## 16. Comparar contactos Boltz-2 vs GNINA

Ahora construiremos un fingerprint geométrico sencillo.

**Residuo en contacto:** algún átomo pesado del residuo está dentro de una distancia máxima del ligando.

Usaremos inicialmente:

**`contact_cutoff = 4.5 Å`**

Este es el valor inicial recomendado para una comparación local de contactos. Puedes cambiarlo para ver cómo cambia la lista.

La tabla mostrará **todos los residuos detectados** y su distancia mínima en cada método.

> ### Pregunta para tu equipo
> **Qué contactos aparecen en ambos métodos y cuáles dependen de la pose predicha?**


In [ ]:
#@title Contact fingerprint Boltz-2 vs GNINA

contact_cutoff = 6 #@param {type:"slider", min:3.5, max:6.0, step:0.5}
contact_ligand = "Benzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]

receptor_atoms = []

for line in protein_pdb.splitlines():
    if not line.startswith("ATOM"):
        continue

    element = (
        line[76:78].strip()
        if len(line) >= 78
        else line[12:16].strip()[0]
    )

    if element.upper() == "H":
        continue

    receptor_atoms.append({
        "resi": int(line[22:26]),
        "resname": line[17:20].strip(),
        "xyz": np.array([
            float(line[30:38]),
            float(line[38:46]),
            float(line[46:54])
        ])
    })

def pdb_ligand_heavy_coords(pdb_text, chain="B"):
    xyz = []

    for line in pdb_text.splitlines():
        if not line.startswith(("ATOM", "HETATM")):
            continue
        if len(line) < 54 or line[21].strip() != chain:
            continue

        element = (
            line[76:78].strip()
            if len(line) >= 78
            else line[12:16].strip()[0]
        )

        if element.upper() == "H":
            continue

        xyz.append([
            float(line[30:38]),
            float(line[38:46]),
            float(line[46:54])
        ])

    return np.array(xyz, dtype=float)

def rdkit_heavy_coords(mol):
    conf = mol.GetConformer()

    return np.array([
        [
            conf.GetAtomPosition(i).x,
            conf.GetAtomPosition(i).y,
            conf.GetAtomPosition(i).z
        ]
        for i, atom in enumerate(mol.GetAtoms())
        if atom.GetAtomicNum() > 1
    ])

def contact_residues(lig_xyz, cutoff):
    contacts = {}

    if len(lig_xyz) == 0:
        return contacts

    for atom in receptor_atoms:
        d = np.linalg.norm(lig_xyz - atom["xyz"], axis=1).min()

        if d <= cutoff:
            key = (atom["resname"], atom["resi"])
            contacts[key] = min(d, contacts.get(key, 999.0))

    return contacts

boltz_xyz = pdb_ligand_heavy_coords(
    aligned_boltz[contact_ligand],
    "B"
)
gnina_xyz = rdkit_heavy_coords(
    gnina_best_mols[contact_ligand]
)

boltz_contacts = contact_residues(
    boltz_xyz,
    contact_cutoff
)
gnina_contacts = contact_residues(
    gnina_xyz,
    contact_cutoff
)

B = set(boltz_contacts)
G = set(gnina_contacts)

all_contacts = sorted(B | G, key=lambda x: x[1])

rows = []

for residue in all_contacts:
    resname, resi = residue

    if residue in B and residue in G:
        categoria = "Compartido"
    elif residue in B:
        categoria = "Solo Boltz-2"
    else:
        categoria = "Solo GNINA"

    rows.append({
        "Residuo": f"{resname}{resi}",
        "Categoría": categoria,
        "Distancia Boltz-2 (Å)": round(boltz_contacts[residue], 2) if residue in B else np.nan,
        "Distancia GNINA (Å)": round(gnina_contacts[residue], 2) if residue in G else np.nan
    })

contact_table = pd.DataFrame(rows)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", None
):
    display(contact_table)

shared = sorted(B & G, key=lambda x: x[1])
boltz_only = sorted(B - G, key=lambda x: x[1])
gnina_only = sorted(G - B, key=lambda x: x[1])

union = len(B | G)
jaccard = len(B & G) / union if union else np.nan

print(f"Contactos compartidos: {len(shared)}")
print(f"Contactos solo de Boltz-2: {len(boltz_only)}")
print(f"Contactos solo de GNINA: {len(gnina_only)}")

if np.isfinite(jaccard):
    print(f"Índice de Jaccard de los contactos: {jaccard:.2f}")
else:
    print("No se encontraron contactos con este cutoff.")

print("El índice de Jaccard mide acuerdo entre las listas de contactos, no exactitud.")


In [ ]:
#@title Visualizar todos los contactos detectados

contact_view_ligand = contact_ligand

shared_res = [num for res, num in shared]
boltz_only_res = [num for res, num in boltz_only]
gnina_only_res = [num for res, num in gnina_only]
all_contact_res = sorted(set(shared_res + boltz_only_res + gnina_only_res))

view = py3Dmol.view(width=900, height=620)
view.setBackgroundColor("white")
view.addModel(protein_pdb, "pdb")
view.setStyle(
    {"model":0},
    {"cartoon":{"color":"lightgray", "opacity":0.40}}
)

if shared_res:
    view.setStyle(
        {"model":0, "chain":"A", "resi":shared_res},
        {"stick":{"colorscheme":"purpleCarbon", "radius":0.28}}
    )

if boltz_only_res:
    view.setStyle(
        {"model":0, "chain":"A", "resi":boltz_only_res},
        {"stick":{"colorscheme":"greenCarbon", "radius":0.24}}
    )

if gnina_only_res:
    view.setStyle(
        {"model":0, "chain":"A", "resi":gnina_only_res},
        {"stick":{"colorscheme":"orangeCarbon", "radius":0.24}}
    )

view.addModel(
    aligned_boltz[contact_view_ligand],
    "pdb"
)
view.setStyle(
    {"model":1, "chain":"B"},
    {"stick":{"colorscheme":"greenCarbon", "radius":0.28}}
)

view.addModel(
    Chem.MolToMolBlock(
        gnina_best_mols[contact_view_ligand]
    ),
    "sdf"
)
view.setStyle(
    {"model":2},
    {"stick":{"colorscheme":"orangeCarbon", "radius":0.28}}
)

# Etiquetas para todos los residuos de contacto.
residue_xyz = {}

for residue in B | G:
    matching = [
        atom["xyz"]
        for atom in receptor_atoms
        if (atom["resname"], atom["resi"]) == residue
    ]
    if matching:
        residue_xyz[residue] = np.mean(np.array(matching), axis=0)

for residue, xyz in residue_xyz.items():
    resname, resi = residue

    if residue in B and residue in G:
        bg = "purple"
    elif residue in B:
        bg = "green"
    else:
        bg = "orange"

    view.addLabel(
        f"{resname}{resi}",
        {
            "position":{
                "x":float(xyz[0]),
                "y":float(xyz[1]),
                "z":float(xyz[2])
            },
            "fontColor":"white",
            "backgroundColor":bg,
            "fontSize":10,
            "showBackground":True
        }
    )

if all_contact_res:
    view.zoomTo(
        {"model":0, "chain":"A", "resi":all_contact_res}
    )
else:
    view.zoomTo({"model":2})

view.show()

print("Morado: contacto en ambos métodos.")
print("Verde: contacto solo en Boltz-2.")
print("Naranja: contacto solo en GNINA.")


## 17. Explorar en 3D y clasificar interacciones con ProLIF

Un score resume una pose en un número, pero no explica por sí solo **qué residuos interactúan con el ligando**.

ProLIF identifica interacciones proteína-ligando y puede mostrarlas directamente sobre la estructura 3D.

En esta actividad buscaremos **todos los tipos de interacción disponibles en ProLIF**, no solo contactos hidrofóbicos. Dependiendo de la química del ligando y del pocket pueden aparecer, por ejemplo:

- Hydrophobic
- VdWContact
- HBDonor
- HBAcceptor
- PiStacking
- FaceToFace
- EdgeToFace
- interacciones iónicas
- interacciones con halógenos
- interacciones con metales

Para Benzene, Toluene y n-Propylbenzene es razonable esperar que predominen contactos hidrofóbicos, van der Waals y posibles interacciones aromáticas.

Primero inspecciona la pose sin ver la respuesta. Después ejecuta ProLIF y compara tu predicción con la tabla y la visualización 3D.

> ### Pregunta para tu equipo
> **Mirando solamente la estructura 3D, qué residuos e interacciones creen que están estabilizando esta pose?**


In [ ]:
#@title Explorar en 3D + tabla de interacciones ProLIF

run_prolif = True #@param {type:"boolean"}
prolif_ligand = "Benzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]

mol = gnina_best_mols[prolif_ligand]
sdf_block = Chem.MolToMolBlock(mol)

# 1. Inspección visual antes de mostrar las interacciones.
view = py3Dmol.view(width=900, height=560)
view.setBackgroundColor("white")
view.addModel(protein_pdb, "pdb")
view.setStyle(
    {"model":0},
    {"cartoon":{"color":"lightgray"}}
)
view.addSurface(
    py3Dmol.VDW,
    {"opacity":0.10, "color":"#D9D9D9"},
    {"model":0}
)

view.addModel(sdf_block, "sdf")
view.setStyle(
    {"model":1},
    {
        "stick":{"colorscheme":"orangeCarbon", "radius":0.30},
        "sphere":{"colorscheme":"orangeCarbon", "scale":0.18}
    }
)

view.zoomTo({"model":1})
view.show()

print("Ligando: naranja.")
print("Antes de ver la tabla, discutan qué residuos e interacciones esperan encontrar.")

if run_prolif:
    try:
        ensure_package("prolif==2.2.1", "prolif")
        import prolif as plf
        from prolif.plotting.complex3d import Complex3D

        prot_rd = Chem.MolFromPDBBlock(
            protein_pdb,
            removeHs=False,
            sanitize=False,
            proximityBonding=True
        )

        if prot_rd is None:
            raise RuntimeError("RDKit no pudo leer la proteína.")

        # Intentar una sanitización parcial sin detener el análisis si alguna
        # operación no es compatible con el PDB.
        try:
            Chem.SanitizeMol(
                prot_rd,
                sanitizeOps=(
                    Chem.SanitizeFlags.SANITIZE_FINDRADICALS
                    | Chem.SanitizeFlags.SANITIZE_SETAROMATICITY
                    | Chem.SanitizeFlags.SANITIZE_SETCONJUGATION
                    | Chem.SanitizeFlags.SANITIZE_SETHYBRIDIZATION
                )
            )
        except Exception:
            pass

        prot_plf = plf.Molecule.from_rdkit(prot_rd)
        lig_plf = plf.Molecule.from_rdkit(mol)

        all_interactions = plf.Fingerprint.list_available()

        fp = plf.Fingerprint(
            interactions=all_interactions,
            count=True,
            vicinity_cutoff=6.0
        )

        fp.run_from_iterable(
            [lig_plf],
            prot_plf,
            progress=False
        )

        fp_df = fp.to_dataframe()

        interactions = []

        if not fp_df.empty:
            row = fp_df.iloc[0]

            for col, value in row.items():
                try:
                    count = int(value)
                except Exception:
                    count = 1 if bool(value) else 0

                if count <= 0:
                    continue

                if isinstance(col, tuple):
                    residue = str(col[-2]) if len(col) >= 2 else "Residuo"
                    interaction = str(col[-1])
                else:
                    residue = "Residuo"
                    interaction = str(col)

                interactions.append({
                    "Residuo": residue,
                    "Interacción": interaction,
                    "N": count
                })

        interaction_table = (
            pd.DataFrame(interactions)
            .drop_duplicates()
            .sort_values(["Residuo", "Interacción"])
            .reset_index(drop=True)
            if interactions
            else pd.DataFrame(
                columns=["Residuo", "Interacción", "N"]
            )
        )

        if interaction_table.empty:
            print("ProLIF no detectó interacciones con los criterios actuales.")
        else:
            print("Interacciones detectadas por ProLIF:")
            with pd.option_context(
                "display.max_rows", None,
                "display.max_colwidth", None
            ):
                display(interaction_table)

        # Cambiar el color del ligando para que sea fácil de distinguir.
        Complex3D.LIGAND_STYLE = {
            "stick":{
                "colorscheme":"orangeCarbon",
                "radius":0.25
            },
            "sphere":{
                "colorscheme":"orangeCarbon",
                "scale":0.16
            }
        }

        print("Visualización 3D de ProLIF:")
        prolif_view = fp.plot_3d(
            lig_plf,
            prot_plf,
            frame=0,
            display_all=True,
            only_interacting=True,
            remove_hydrogens=True,
            sanitize=False
        )
        display(prolif_view)

        print("Compara la visualización con la predicción que hizo tu grupo.")
        print("Pregunta: qué información aporta ProLIF que el score de docking por sí solo no aporta?")

    except Exception as e:
        print("ProLIF no pudo ejecutarse en este runtime.")
        print("La inspección 3D y la comparación geométrica anterior siguen disponibles.")
        print("Detalle técnico:", str(e)[:500])
else:
    print("Análisis con ProLIF omitido.")


## 18. Actualizar la hipótesis

Ahora tienes varias piezas de evidencia:

- confianza estructural de Boltz-2
- probabilidad de unión de Boltz-2
- Delta G estimada a partir de la predicción de IC50
- `minimizedAffinity` de GNINA
- inspección 3D de las poses
- contactos compartidos y diferentes
- tipos de interacción detectados por ProLIF

> ### Pregunta para tu equipo
> **Después de ver esta nueva evidencia, mantienen la hipótesis del Día 1 o la cambiarían? Qué evidencia tuvo más peso en su decisión?**

No trates los números de Boltz-2 y GNINA como si fueran una medición experimental equivalente. Usa la comparación para identificar **rankings, tendencias, acuerdos, desacuerdos y explicaciones estructurales**.

Al registrar la confianza final del grupo usarás una escala de 1 a 5. Si el grupo todavía está dividido, un valor inicial razonable es **3**.


In [ ]:
#@title Hipótesis al final del Día 2

final_1 = "Toluene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]
final_2 = "Benzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]
final_3 = "n-Propylbenzene" #@param ["Benzene", "Toluene", "n-Propylbenzene"]

evidence = "GNINA principalmente" #@param ["Boltz-2 principalmente", "GNINA principalmente", "Boltz-2 y GNINA juntos", "minimizedAffinity de GNINA", "Interacciones de ProLIF", "Inspección estructural"]
confidence = 2 #@param {type:"slider", min:1, max:5, step:1}

ranking = [final_1, final_2, final_3]

if len(set(ranking)) < 3:
    print("Revisa tu ranking: usa cada ligando una sola vez.")
else:
    display(HTML(f'''
    <div style="border:1px solid #bbb; border-radius:12px; padding:18px; max-width:650px;">
      <h3 style="margin-top:0;">Día 2 | Hipótesis del grupo</h3>
      <p><b>Ranking actual:</b><br>
      1. {final_1}<br>
      2. {final_2}<br>
      3. {final_3}</p>
      <p><b>Evidencia principal:</b> {evidence}</p>
      <p><b>Confianza:</b> {confidence}/5</p>
    </div>
    '''))


## 19. Cierre

Hoy usamos tres niveles de evidencia:

**Boltz-2:** propone un complejo, confianza y afinidad predicha a partir de secuencia + ligando.

**GNINA:** busca poses dentro de una estructura y región definidas y las evalúa con `minimizedAffinity`.

**ProLIF:** describe qué tipos de interacción aparecen entre el ligando y los residuos de la proteína.

> ### Pregunta final
> **Qué conclusión permanece igual entre los métodos y qué evidencia todavía necesitaríamos antes de afirmar que un ligando realmente se une mejor?**

La pregunta que queda para el Día 3 es:

> **Una pose predicha sigue siendo plausible cuando permitimos que el sistema se mueva?**

Todavía no hemos utilizado las estructuras experimentales de los complejos como respuesta.


---

## Herramientas y referencias

- **Boltz-2** - Passaro S, Corso G, Wohlwend J, et al. *Boltz-2: Towards Accurate and Efficient Binding Affinity Prediction.* bioRxiv, 2025. DOI: 10.1101/2025.06.14.659707
- **GNINA** - McNutt AT, Francoeur P, Aggarwal R, et al. *GNINA 1.0: molecular docking with deep learning.* J Cheminform. 2021.
- **Vinardo** - Quiroga R, Villarreal MA. *Vinardo: A Scoring Function Based on Autodock Vina Improves Scoring, Docking, and Virtual Screening.* PLoS ONE. 2016.
- **P2Rank** - Krivák R, Hoksza D. *J Cheminform.* 2018;10:39.
- **ProLIF** - protein-ligand interaction fingerprints.
- **RDKit** y **py3Dmol** - representación molecular y visualización.
- **Cloud-Bind** - colección abierta de notebooks para molecular docking y análisis en Google Colab.
